# Day 2 · 03 · AI/BI Dashboards & Genie — Native Analytics in Databricks

This notebook is the dedicated module for Databricks-native BI: AI/BI Dashboards and Genie Space.
It covers when to use these tools instead of (or alongside) Power BI, how to build a working
dashboard from Gold objects, and how to configure a Genie Space for conversational analytics.

**Duration:** ~30 minutes  
**Format:** UI demos with supporting SQL, followed by a short hands-on task

## Business Scenario

The Sales Director at RetailHub wants to explore KPI data independently without waiting for IT to
publish a new Power BI report. The data team has a well-modelled Gold layer. The question is:
can the business team access it directly without a Power BI setup?

The answer is yes — through AI/BI Dashboards and Genie Space.

## Learning Objectives

By the end of this notebook you can:

- explain when AI/BI Dashboards replace or complement Power BI,
- build a simple AI/BI Dashboard from Gold objects,
- configure a Genie Space with appropriate datasets and instruction notes,
- ask 3 business questions in natural language and evaluate Genie's responses,
- describe Genie's current limitations so participants set realistic expectations.

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.kpi_monthly",
    f"{GOLD}.dim_customer",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    raise Exception("Pre-check failed. Run day1_03_gold_modeling_for_powerbi.ipynb first.")

print("[OK] All required Gold objects exist")
for t in required_tables:
    print(" ", t)

## 1. AI/BI Dashboards vs Power BI — decision framework

Both tools consume the same Gold objects. The decision is about audience, infrastructure, and use case.

| Criterion | AI/BI Dashboard | Power BI |
|---|---|---|
| Setup time | minutes in browser, no install | Power BI Desktop + Gateway + Service |
| Data freshness | always live (SQL on demand) | Import = scheduled, DirectQuery = live |
| Primary audience | Databricks workspace users, analysts | external stakeholders, executive dashboards |
| Unity Catalog security | applied automatically (row/col masks) | must be replicated separately in PBI model |
| Conversational BI | Genie Space — natural language → SQL → chart | Q&A feature (limited) |
| DAX / complex measures | not supported | full DAX engine |
| Embedding in external app | Databricks embed API (limited) | Power BI Embedded (mature, widely used) |
| License cost | included in SQL Warehouse usage | Power BI Premium / Embedded license |

**Decision rule:**

- Use **AI/BI Dashboards** when: the audience has Databricks access, you want live data without refresh scheduling, or you need a fast prototype that non-technical users can explore.
- Use **Power BI** when: the audience is outside Databricks, complex DAX is needed, or reports must be embedded in external applications.
- Use **both**: AI/BI Dashboards for internal self-service and exploration; Power BI for governed executive reporting.

### SQL queries that will back the dashboard

Before building the dashboard, verify that the underlying queries return sensible results.
This is the same Gold layer that Power BI would consume — so the dashboard quality is a direct
reflection of the Gold modeling done in Day 1.

In [ ]:
# Query 1: monthly revenue by category — will be the main trend chart
display(spark.sql(f"""
SELECT
  year_month,
  category,
  SUM(revenue) AS revenue,
  SUM(completed_orders) AS orders
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY year_month, category
ORDER BY year_month, revenue DESC
"""))

In [ ]:
# Query 2: revenue by channel and region — will be the breakdown chart
display(spark.sql(f"""
SELECT
  channel,
  customer_region,
  SUM(revenue) AS revenue,
  ROUND(100.0 * SUM(gross_margin) / NULLIF(SUM(revenue), 0), 2) AS margin_rate_pct
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY channel, customer_region
ORDER BY revenue DESC
"""))

In [ ]:
# Query 3: KPI cards — latest month headline numbers
display(spark.sql(f"""
SELECT
  year_month,
  revenue,
  gross_margin,
  completed_orders,
  ROUND(100.0 * gross_margin / NULLIF(revenue, 0), 2) AS margin_rate_pct
FROM {GOLD}.kpi_monthly
ORDER BY year_month DESC
LIMIT 3
"""))

## 2. `[UI DEMO]` Build an AI/BI Dashboard

This is a live browser demo. Follow these steps:

### Create the dashboard

1. In the Databricks workspace left sidebar: **Dashboards** → **New Dashboard** → **AI/BI Dashboard**.
2. Name: `RetailHub KPI — [participant name]`.

### Add Dataset 1 — monthly revenue by category

1. Click **Add dataset** → choose **SQL** → name it `monthly_by_category`.
2. Paste this query (replace `YOUR_CATALOG` with the actual catalog name from the config cell above):

```sql
SELECT
  year_month,
  category,
  SUM(revenue) AS revenue,
  SUM(completed_orders) AS orders
FROM YOUR_CATALOG.gold.fact_sales_dashboard_monthly
GROUP BY year_month, category
ORDER BY year_month, revenue DESC
```

3. Click **Run** — confirm results.

### Add Dataset 2 — KPI headline numbers

```sql
SELECT
  year_month,
  revenue,
  gross_margin,
  completed_orders,
  ROUND(100.0 * gross_margin / NULLIF(revenue, 0), 2) AS margin_rate_pct
FROM YOUR_CATALOG.gold.kpi_monthly
ORDER BY year_month DESC
```

### Add visualizations

1. **Bar chart:** X = `year_month`, Y = `SUM(revenue)`, Group by = `category`. Title: `Revenue by Category`.
2. **KPI counter:** Value = latest `revenue` from the KPI dataset. Title: `Total Revenue (latest month)`.
3. **Filter widget:** on `category` (multi-select from `monthly_by_category`).

### Share

Click **Publish** to make the dashboard visible to other workspace users with access to the catalog.

**Key observation for participants:** this dashboard is live — it reads Gold every time it opens.
No refresh scheduling, no import, no gateway. The tradeoff: every view hits the SQL Warehouse.

## 3. `[UI DEMO]` Genie Space — conversational BI

Genie Space is the "ask a question in natural language, get a chart" capability.
Under the hood, Genie writes and executes SQL against your Gold objects.

### Configure a Genie Space

1. Left sidebar → **Genie** (or search for it).
2. **Create a space** → Name: `RetailHub KPI Explorer`.
3. **Add data:** select these tables (use the full 3-part name: catalog.schema.table):
   - `fact_sales_dashboard_monthly` — primary KPI source
   - `kpi_monthly` — headline KPI numbers
   - `dim_customer` — for region and segment filtering

4. **Add instruction note** (this is the most important configuration step — tell Genie about your data):

```
This Genie Space covers RetailHub retail sales data.

Data definitions:
- Revenue and orders in fact_sales_dashboard_monthly are already filtered to Completed status only.
- year_month is in 'YYYY-MM' format. Use it for trend questions.
- customer_region represents the geographic region of the buyer.
- channel values: 'Online', 'Store', 'Partner'.
- category is the product category (Electronics, Clothing, Food, etc.).
- gross_margin = revenue - cost; margin_rate_pct = gross_margin / revenue * 100.

Common questions:
- "Which region has the highest revenue?" → use fact_sales_dashboard_monthly, GROUP BY customer_region.
- "What is the trend for Electronics?" → GROUP BY year_month WHERE category = 'Electronics'.
- "Compare Online vs Store margin" → GROUP BY channel.
```

5. Click **Save** → **Start chatting**.

### Demo questions to ask (live, in Polish and English)

Ask these questions in sequence and show participants the generated SQL + chart:

| Question | Expected Genie behavior |
|---|---|
| `"Który region miał najwyższy przychód w ostatnim roku?"` | GROUP BY customer_region, bar chart |
| `"Show me revenue trend for Electronics over time."` | WHERE category = 'Electronics', line chart by year_month |
| `"Porównaj marżę dla kanału Online i Store."` | GROUP BY channel, compare gross_margin or margin_rate_pct |

**For each answer, click "Show SQL"** — participants see that Genie writes real SQL, not magic.
This is important: the Gold layer quality directly determines Genie quality.

### What to demonstrate when Genie makes a mistake

Try: `"Jaki był przychód w Q1 2025 dla każdego segmentu klientów?"`

Genie may need to join `fact_sales_dashboard_monthly` with `dim_customer` to get segment.
If the answer is wrong or missing, show participants:
1. Click **Edit** → update the instruction note to clarify the segment → customer relationship.
2. Or: add a pre-joined Gold view as a dataset (e.g. `fact_sales_dashboard_monthly` already has segment if it was modeled with it).

**Lesson:** Genie is an amplifier of a good data model, not a replacement for it.

## 4. Genie current limitations

Be explicit with participants. Overselling Genie leads to production disappointment.

| Limitation | Impact | Mitigation |
|---|---|---|
| Complex window functions (YoY, rolling averages) | may generate incorrect SQL | pre-compute these in a Gold view, add it as a Genie dataset |
| Multi-hop joins across 4+ tables | results can be wrong | use pre-joined Gold tables as Genie datasets |
| Highly domain-specific terminology | Genie misinterprets terms | write detailed instruction notes with glossary |
| Result charts are basic (bar, line, table) | cannot replicate PBI formatting | Genie is for exploration, not polished reports |
| Large result sets truncated | analyst may not see all rows | Genie is not a data export tool |
| No DAX-equivalent measures | calculated KPI logic must live in Gold SQL | design Gold to expose pre-calculated metrics |
| Workspace permission required | external stakeholders cannot use Genie | Power BI is the right tool for external distribution |

**Honest summary:** Genie is excellent for self-service exploration by analysts who already understand the data. It is not a replacement for governed Power BI reports used by executives or embedded in external products.

## 5. Hands-on: compare AI/BI Dashboard vs Power BI source objects

This short lab ties together the performance concepts from `day2_02` and the
AI/BI concepts from this notebook.

**Task:** measure the response time difference between querying the detail table vs the aggregate table,
then decide which one should back the Genie Space and which one should back the AI/BI Dashboard.

Run both queries below, then open **Query History** and compare execution times.

In [ ]:
# Source 1: detail-grain table — every order line for the last 24 months
import time

t0 = time.time()
result_detail = spark.sql(f"""
SELECT
  date_format(order_date, 'yyyy-MM') AS year_month,
  category,
  SUM(line_revenue) AS revenue
FROM {GOLD}.fact_sales_dashboard
WHERE is_completed
  AND order_date >= add_months(current_date(), -24)
GROUP BY date_format(order_date, 'yyyy-MM'), category
ORDER BY year_month, revenue DESC
""")
result_detail.count()  # force execution
t_detail = round(time.time() - t0, 2)

print(f"Detail-grain query time: {t_detail}s")
display(result_detail.limit(10))

In [ ]:
# Source 2: pre-aggregated monthly table — same answer, much smaller scan
t0 = time.time()
result_agg = spark.sql(f"""
SELECT
  year_month,
  category,
  SUM(revenue) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY year_month, category
ORDER BY year_month, revenue DESC
""")
result_agg.count()  # force execution
t_agg = round(time.time() - t0, 2)

print(f"Aggregate table query time: {t_agg}s")
display(result_agg.limit(10))

In [ ]:
# Summary comparison
display(spark.createDataFrame([
    ("Detail grain (fact_sales_dashboard)",   t_detail, "every order line"),
    ("Monthly aggregate (fact_sales_dashboard_monthly)", t_agg,   "pre-aggregated by month"),
], ["Source", "Query time (s)", "Grain"]))

print("""
Discussion:
- Both return the same business answer (revenue by month by category).
- The aggregate table is faster because it scans far fewer rows.
- For an AI/BI Dashboard or Genie Space used by 20+ analysts simultaneously,
  the aggregate version costs 10-100x less per query.
- Use the aggregate table as the primary Genie dataset.
  Use the detail view only for drill-through when a user explicitly asks for line-level data.
""")

## 6. Recommended architecture: AI/BI + Power BI together

These tools are not competitors — they serve different parts of the analytics value chain.

```
                         Unity Catalog Gold Layer
                                   │
          ┌────────────────────────┼──────────────────────┐
          │                        │                       │
  fact_sales_dashboard_monthly   kpi_monthly    v_fact_sales_incremental
    (aggregate, small, fast)   (headline KPI)   (detail view, bounded)
          │                        │                       │
  ┌───────┴───────┐        ┌───────┴───────┐     ┌────────┴────────┐
  │  AI/BI        │        │   Genie Space │     │   Power BI      │
  │  Dashboard    │        │               │     │   Import mode   │
  │               │        │  ask questions│     │                 │
  │ live, fast,   │        │  in Polish    │     │ scheduled,      │
  │ no refresh    │        │  or English   │     │ external users  │
  └───────────────┘        └───────────────┘     └─────────────────┘
  Internal analysts         Self-service BI        Executive reports,
  Data team review          Business users          embedded reports
```

**The Gold layer is the single source of truth for all three consumers.**
Update the Gold model once → all three surfaces reflect it automatically.

## Summary

After this notebook:

- You know when AI/BI Dashboards replace or complement Power BI.
- You can build a simple AI/BI Dashboard from Gold SQL queries.
- You can configure a Genie Space with instruction notes for better answers.
- You understand Genie's current limitations and can set realistic expectations.
- You have seen that the aggregate Gold table is significantly faster than the detail table
  as a Genie/AI/BI dataset — reinforcing the Gold modeling decisions from Day 1.

This module completes the RetailHub analytics story:

**Data → Delta → Gold → Power BI + AI/BI Dashboards + Genie**